# Training semantic segmentation (DeepLabv3+ or PSPNet) in Google Colab with CVAT masks and Azure Blob storage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasantana/sam-image-processing/blob/main/notebooks/semantic_segmentation_training.ipynb)

In [ ]:
# 🔧 Install Dependencies
# Run this cell first when opening the notebook in Google Colab

!pip install azure-storage-blob>=12.17.0
!pip install opencv-python>=4.8.0  
!pip install pillow>=10.0.0
!pip install numpy>=1.24.0
!pip install matplotlib>=3.7.0

print("🎉 Dependencies installed successfully!")

# 🛰️ Semantic Segmentation Training Setup

This notebook initializes the connection to Azure Blob Storage, downloads drone imagery and labeled masks, and prepares reproducible training and validation datasets for semantic segmentation using DeepLabv3+ or PSPNet.

**Dataset structure:**
- Images: `assets/drone-imagery`
- Masks: `assets/pascual_annotation_masks/SegmentationClass`
- Label map: `assets/pascual_annotation_masks/labelmap.txt`

**Azure Blob Storage:**
- Container: `general-purpose`
- Connection string: stored securely in `AZURE_CONN_STR`

## 1: Azure Blob Connection & Download

In [ ]:
# 📦 Connect to Azure Blob Storage and download images and masks
from azure.storage.blob import BlobServiceClient
import os

AZURE_CONN_STR = "DefaultEndpointsProtocol=...<place complete string here>"
CONTAINER_NAME = "general-purpose"

LOCAL_IMG_DIR = "assets/drone-imagery"
LOCAL_MASK_DIR = "assets/pascual_annotation_masks/SegmentationClass"
LOCAL_LABELMAP_DIR = "assets/pascual_annotation_masks"

os.makedirs(LOCAL_IMG_DIR, exist_ok=True)
os.makedirs(LOCAL_MASK_DIR, exist_ok=True)

# Connect to blob container
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

def download_blobs(prefix, local_dir, ext):
    blobs = container_client.list_blobs(name_starts_with=prefix)
    for blob in blobs:
        if blob.name.endswith(ext):
            local_path = os.path.join(local_dir, os.path.basename(blob.name))
            if not os.path.exists(local_path):
                with open(local_path, "wb") as f:
                    f.write(container_client.download_blob(blob.name).readall())
    print(f"Downloaded files from {prefix} to {local_dir}")

# Download images and masks
download_blobs("images/all/", LOCAL_IMG_DIR, ".jpg")
download_blobs("masks/all/", LOCAL_MASK_DIR, ".png")

## 2: 🧪 Dataset Preparation

This section reads and sorts the downloaded images and masks, then creates a reproducible split:
- First 53 images → Training set
- Remaining images → Validation set

Each image is matched with its corresponding mask by filename.

In [ ]:
import os

# List and sort image and mask files
image_files = sorted([f for f in os.listdir(LOCAL_IMG_DIR) if f.endswith(".jpg")])
mask_files = sorted([f for f in os.listdir(LOCAL_MASK_DIR) if f.endswith(".png")])

# Match image-mask pairs
pairs = [(img, img.replace(".jpg", ".png")) for img in image_files if img.replace(".jpg", ".png") in mask_files]

# Reproducible split: first 53 for training
train_pairs = pairs[:53]
val_pairs = pairs[53:]

# Build full path dictionaries
def build_dataset(pairs, img_dir, mask_dir):
    return [
        {
            "img_path": os.path.join(img_dir, img),
            "seg_map_path": os.path.join(mask_dir, mask)
        }
        for img, mask in pairs
    ]

train_dataset = build_dataset(train_pairs, LOCAL_IMG_DIR, LOCAL_MASK_DIR)
val_dataset = build_dataset(val_pairs, LOCAL_IMG_DIR, LOCAL_MASK_DIR)

print(f"✅ Train set: {len(train_dataset)} images")
print(f"✅ Val set: {len(val_dataset)} images")

## 3: 🚀 Next Steps

Now that the datasets are prepared, we’ll:
1. Define the class map from `labelmap.txt`
2. Configure the MMSegmentation training pipeline
3. Train DeepLabv3+ or PSPNet using the prepared dataset
4. Evaluate performance and export the model for deployment